# SABR fit validation

This notebook is a **research / QA workbench** for the SABR calibration and the Breeden–Litzenberger risk-neutral density (RND) extraction that power the dashboard. It is **not** the deliverable — the deliverable is the FastAPI service in `../backend`. Use this notebook to:

1. Sanity-check the Hagan lognormal SABR approximation against known limits.
2. Confirm the RND integrates to 1 and is arbitrage-consistent (non-negative density, monotone CDF).
3. Validate the risk-neutral probability against a closed-form Black benchmark for a flat smile.
4. Eyeball fit quality on each asset class using the mock surfaces (swap in a live `xbbg` chain when on the Terminal).

When run on the Terminal machine, replace the `MockProvider` cell with a live `BloombergProvider` and the same code validates against real market smiles.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../backend'))
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

from app.core.sabr import calibrate_sabr, sabr_lognormal_vol, sabr_shifted_vol
from app.core.breeden_litzenberger import extract_rnd, black76_call
from app.data.bloomberg import MockProvider, ASSET_DEFAULTS
from app.data.chain_builder import build_chain
print('imports OK')

## 1. Flat-smile benchmark: RND must match analytic lognormal

For a flat vol (ν→0, β=1) the risk-neutral law of $S_T$ is lognormal and, under the forward measure with $r=0$, $\mathbb{P}(S_T>K)=N(d_2)$. The Breeden–Litzenberger PDF should reproduce this exactly.

In [ ]:
F, T, vol = 100.0, 0.5, 0.20
strikes = np.linspace(70, 140, 15)
mkt = np.full_like(strikes, vol)
params = calibrate_sabr(F, strikes, mkt, T, beta=1.0)
rnd = extract_rnd(params, r=0.0, n_grid=1500)

Ks = np.linspace(75, 135, 25)
num = np.array([rnd.prob_above(k) for k in Ks])
d2 = (np.log(F/Ks) - 0.5*vol**2*T)/(vol*np.sqrt(T))
ana = norm.cdf(d2)

fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].plot(rnd.strikes, rnd.pdf); ax[0].set_title('RND PDF (flat smile)'); ax[0].set_xlabel('S_T')
ax[1].plot(Ks, ana, 'k-', label='Analytic N(d2)')
ax[1].plot(Ks, num, 'r--', label='Breeden-Litzenberger')
ax[1].legend(); ax[1].set_title('P(S_T > K): analytic vs numeric'); ax[1].set_xlabel('K')
plt.tight_layout(); plt.show()
print('max abs error in P(S>K):', np.max(np.abs(ana-num)))
print('PDF integral:', np.trapezoid(rnd.pdf, rnd.strikes))
print('E[S_T] (should equal F):', rnd.mean())

## 2. Per-asset-class smile fits on mock surfaces

Calibrate SABR to each asset class and overlay the fitted smile on the market points. Watch for: fit RMSE (should be a few tenths of a vol point), sign of ρ (negative for equity skew), and a well-behaved RND (single-peaked, integrates to 1).

In [ ]:
prov = MockProvider()
cases = ['SPX Index', 'AAPL US Equity', 'EURUSD Curncy', 'XAU Curncy', 'FEDFUNDS']
fig, axes = plt.subplots(len(cases), 2, figsize=(13, 3.2*len(cases)))
for i, und in enumerate(cases):
    chain = build_chain(prov, und)
    sl = chain.expiries[len(chain.expiries)//2]
    K, v = sl.smile()
    beta = ASSET_DEFAULTS[chain.asset_class]['beta']
    p = calibrate_sabr(sl.forward, K, v, sl.T, beta=beta, shift=chain.shift)
    Kd = np.linspace(K.min(), K.max(), 200)
    axes[i,0].plot(K, v*100, 'o', ms=4, label='market')
    axes[i,0].plot(Kd, p.vol(Kd)*100, '-', label=f'SABR (rmse={p.rmse*100:.2f}bp/100)')
    axes[i,0].set_title(f'{und}  [{chain.asset_class}]  ρ={p.rho:.2f} ν={p.nu:.2f}')
    axes[i,0].set_xlabel('strike'); axes[i,0].set_ylabel('IV %'); axes[i,0].legend(fontsize=8)
    if chain.asset_class == 'RATES':
        lo, hi = sl.forward-4, sl.forward+4
    else:
        lo, hi = K.min()*0.5, K.max()*1.6
    rnd = extract_rnd(p, strike_lo=lo, strike_hi=hi, n_grid=1200)
    axes[i,1].plot(rnd.strikes, rnd.pdf)
    axes[i,1].set_title(f'RND  integral={np.trapezoid(rnd.pdf, rnd.strikes):.3f}  median={rnd.quantile(0.5):.2f}')
    axes[i,1].set_xlabel('level')
plt.tight_layout(); plt.show()

## 3. Arbitrage checks

A valid RND requires: (a) non-negative density everywhere, (b) monotone non-decreasing CDF in [0,1], (c) call prices convex and monotone decreasing in strike (butterfly / call-spread no-arbitrage). We verify these on the fitted curve.

In [ ]:
chain = build_chain(prov, 'SPX Index')
sl = chain.nearest_expiry(chain.as_of.replace(year=chain.as_of.year+1))
K, v = sl.smile()
p = calibrate_sabr(sl.forward, K, v, sl.T, beta=1.0)
rnd = extract_rnd(p, strike_lo=K.min()*0.4, strike_hi=K.max()*1.7, n_grid=1500)
print('min pdf (>=0?):', rnd.pdf.min())
print('cdf monotone?:', bool(np.all(np.diff(rnd.cdf) >= -1e-9)))
print('cdf endpoints:', rnd.cdf[0], rnd.cdf[-1])
dC = np.diff(rnd.call_prices)
print('calls decreasing in K?:', bool(np.all(dC <= 1e-6)))
d2C = np.diff(rnd.call_prices, 2)
print('calls convex in K (butterfly>=0)?:', bool(np.all(d2C >= -1e-6)))

## 4. Going live on the Terminal

On the Bloomberg machine, swap the provider and everything else is identical:

```python
from app.data.bloomberg import BloombergProvider
prov = BloombergProvider()          # uses xbbg; blpapi fallback
assert prov.available, 'start the Terminal / check DAPI on :8194'
chain = build_chain(prov, 'SPX Index US 12/18/26 C')   # or an OPT_CHAIN root
```

The service, API and frontend need no changes — `get_provider()` auto-selects the live source when `xbbg`/`blpapi` can connect.